In [1]:
import json, random
from pathlib import Path
import numpy as np, torch, torch.nn as nn
from torch.utils.data import DataLoader
from scipy.stats import spearmanr
from tqdm.auto import tqdm

cwd  = Path.cwd()
REPO = cwd if (cwd / "core").exists() else cwd.parent
OUT  = REPO / "python/layerwise"
LAYERS = [14, 15, 16, 17]

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_float32_matmul_precision("high")
print("device:", DEVICE, "| data:", OUT)


device: mps | data: /Users/bitan/Desktop/aperdiff/python/layerwise


/Users/bitan/Desktop/aperdiff/python/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TODO: This is actually an older <clever> version of what we finally go with. In effect, a Transformer encoder overfits, and we use MLP instead (cheaper compute, similar results)

In [2]:
class DiffNet(nn.Module):
    def __init__(self, d_in=4096, d_model=512, nhead=8, layers=2, ff=2048, dropout=0.1):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(d_in, d_model), nn.LayerNorm(d_model), nn.GELU())
        enc = nn.TransformerEncoderLayer(
            d_model, nhead, dim_feedforward=ff, dropout=dropout,
            batch_first=True, norm_first=True, activation="gelu")
        self.enc = nn.TransformerEncoder(enc, layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, 128), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(128, 1))

    def forward(self, x, pad_mask=None):
        h = self.proj(x)
        h = self.enc(h, src_key_padding_mask=pad_mask)
        return self.head(h).squeeze(-1)


In [3]:
def load_entries(path):
    return torch.load(path, map_location="cpu", weights_only=False)

def load_role(role, l):
    entries = []
    for p in sorted(OUT.glob(f"{role}_*_L{l}.pt")):
        entries += load_entries(p)
    return entries

def sequitize(entries):
    seqs = []
    for e in entries:
        for k in ("a", "b"):
            x = e[f"feats_{k}"].float()
            y = torch.tensor(e[f"labels_{k}"], dtype=torch.float32)
            if x.shape[0] == y.shape[0]:
                seqs.append((x, y))
    return seqs

def collate(batch):
    L = max(x.shape[0] for x, _ in batch)
    X = torch.zeros(len(batch), L, 4096)
    Y = torch.full((len(batch), L), -1.0)
    P = torch.ones(len(batch), L, dtype=torch.bool)
    for i, (x, y) in enumerate(batch):
        n = x.shape[0]
        X[i, :n] = x; Y[i, :n] = y; P[i, :n] = False
    return X, Y, P

def loader(entries, shuffle=False, bs=4):
    return DataLoader(sequitize(entries), batch_size=bs, shuffle=shuffle, collate_fn=collate)

def masked_bce(logits, y, pad):
    loss = nn.functional.binary_cross_entropy_with_logits(logits, y.clamp(0, 1), reduction="none")
    m = (~pad) & (y >= 0)
    return loss[m].mean()

@torch.no_grad()
def evaluate(model, dl):
    model.eval(); ps, gs = [], []
    for X, Y, P in dl:
        out = torch.sigmoid(model(X.to(DEVICE), P.to(DEVICE))).cpu()
        for i in range(X.shape[0]):
            m = (~P[i]) & (Y[i] >= 0)
            ps.append(out[i][m].numpy()); gs.append(Y[i][m].numpy())
    p = np.concatenate(ps); g = np.concatenate(gs)
    rho = spearmanr(p, g).statistic
    clear = (g <= 0.05) | (g >= 0.5)
    acc = float(((p[clear] > 0.5) == (g[clear] >= 0.5)).mean()) if clear.any() else float("nan")
    return rho, acc


In [5]:
def train_one(l, epochs=20, lr=3e-4, bs=4):
    torch.manual_seed(SEED)
    model = DiffNet().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    tr = loader(load_role("train", l), shuffle=True, bs=bs)
    va = loader(load_role("val", l), bs=bs)
    ckpt = OUT / f"probe_L{l}.pth"
    best = -1.0
    bar = tqdm(range(1, epochs + 1), desc=f"L{l}", ncols=110, leave=False)
    for ep in bar:
        model.train(); tot = 0.0
        for X, Y, P in tr:
            opt.zero_grad()
            loss = masked_bce(model(X.to(DEVICE), P.to(DEVICE)), Y.to(DEVICE), P.to(DEVICE))
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tot += float(loss)
        sched.step()
        loss = tot / len(tr)
        rho, acc = evaluate(model, va)
        if rho > best:
            best = rho; torch.save(model.state_dict(), ckpt)
        bar.set_postfix(loss=f"{loss:.4f}", acc=f"{acc:.3f}", val_rho=f"{rho:.4f}", best=f"{best:.4f}")
    print(f"L{l}: best val rho={best:.4f}  ->  {ckpt.name}")
    return best


In [6]:
train_one(14)


/var/folders/92/p9w4rns94gs4rlfs009w9zrc0000gn/T/ipykernel_54943/3671758863.py:8: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(enc, layers)
L14:   0%|                                                                             | 0/20 [00:00<?, ?it/s]/var/folders/92/p9w4rns94gs4rlfs009w9zrc0000gn/T/ipykernel_54943/2008472184.py:17: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:821.)
  tot += float(loss)


L14: best val rho=0.3745  ->  probe_L14.pth


np.float64(0.3745302651823501)

In [7]:
train_one(15)


/var/folders/92/p9w4rns94gs4rlfs009w9zrc0000gn/T/ipykernel_54943/3671758863.py:8: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(enc, layers)


L15: best val rho=0.3852  ->  probe_L15.pth


np.float64(0.3851621839608525)

In [8]:
train_one(16)


/var/folders/92/p9w4rns94gs4rlfs009w9zrc0000gn/T/ipykernel_54943/3671758863.py:8: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(enc, layers)


L16: best val rho=0.3812  ->  probe_L16.pth


np.float64(0.38120857174141415)

In [9]:
train_one(17)


/var/folders/92/p9w4rns94gs4rlfs009w9zrc0000gn/T/ipykernel_54943/3671758863.py:8: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(enc, layers)


L17: best val rho=0.3838  ->  probe_L17.pth


np.float64(0.3838337113622496)

In [10]:
@torch.no_grad()
def score_ckpt(l, role, bs=4):
    m = DiffNet().to(DEVICE)
    m.load_state_dict(torch.load(OUT / f"probe_L{l}.pth", map_location=DEVICE)); m.eval()
    return evaluate(m, loader(load_role(role, l), bs=bs))

rows = [(l, *score_ckpt(l, "val"), *score_ckpt(l, "syn")) for l in LAYERS]   # val=holdout, syn=OOD

print("| Layer | val rho | val acc | OOD rho | OOD acc |")
print("|--:|--:|--:|--:|--:|")
for l, vrho, vacc, srho, sacc in rows:
    print(f"| {l} | {vrho:.4f} | {vacc:.4f} | {srho:.4f} | {sacc:.4f} |")

best = max(rows, key=lambda r: r[3])
print(f"\nbest OOD: L{best[0]} (synthetic rho={best[3]:.4f})")


/var/folders/92/p9w4rns94gs4rlfs009w9zrc0000gn/T/ipykernel_54943/3671758863.py:8: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(enc, layers)


| Layer | val rho | val acc | OOD rho | OOD acc |
|--:|--:|--:|--:|--:|
| 14 | 0.3745 | 0.9214 | 0.1411 | 0.8586 |
| 15 | 0.3852 | 0.9215 | 0.1622 | 0.8570 |
| 16 | 0.3812 | 0.9215 | 0.1644 | 0.8568 |
| 17 | 0.3838 | 0.9227 | 0.1649 | 0.8569 |

best OOD: L17 (synthetic rho=0.1649)


In [11]:
best_layer = best[0]
model = DiffNet().to(DEVICE)
model.load_state_dict(torch.load(OUT / f"probe_L{best_layer}.pth", map_location=DEVICE)); model.eval()

rec = load_role("syn", best_layer)[0]
with torch.no_grad():
    for k in ("a", "b"):
        x = rec[f"feats_{k}"].float().unsqueeze(0).to(DEVICE)
        pred = torch.sigmoid(model(x)).squeeze(0).cpu().numpy()
        print(f"-- {rec['id']} side {k} (layer {best_layer}) --")
        for w, pr, gd in list(zip(rec[f"tokens_{k}"], pred, rec[f"labels_{k}"]))[:15]:
            print(f"   {pr:5.2f}  gold {gd:>5}  {w}")


/var/folders/92/p9w4rns94gs4rlfs009w9zrc0000gn/T/ipykernel_54943/3671758863.py:8: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(enc, layers)


-- syn_de-fr_1 side a (layer 17) --
    0.02  gold   0.0  Die
    0.05  gold   1.0  Energieetikette
    0.03  gold   0.0  für
    0.15  gold   0.0  Raumklimageräte
    0.02  gold   0.0  Die
    0.03  gold   1.0  Energieetikette
    0.02  gold   0.0  bietet
    0.01  gold   0.0  Informationen
    0.01  gold   0.0  zur
    0.02  gold   0.0  Energieeffizienz
    0.03  gold   0.0  der
    0.07  gold   0.0  damit
    0.19  gold   0.0  deklarierten
    0.05  gold   0.0  Geräte
    0.02  gold   0.0  .
-- syn_de-fr_1 side b (layer 17) --
    0.04  gold   0.0  L’étiquette
    0.07  gold   1.0  -
    0.08  gold   0.0  énergie
    0.02  gold  -1.0  pour
    0.02  gold   0.0  les
    0.07  gold   0.0  climatiseurs
    0.02  gold   1.0  L
    0.03  gold  -1.0  '
    0.03  gold   0.0  étiquette
    0.02  gold   0.0  -
    0.03  gold  -1.0  énergie
    0.06  gold   0.0  d
    0.11  gold   0.0  '
    0.25  gold   0.0  un
    0.42  gold  -1.0  appareil
